### Ventures AI

Chat Bot To query data from [Y-Combinator Startup directory](https://www.ycombinator.com/companies)
> Data Source: https://github.com/yc-oss/api (open sourec Y Combinator companies API)

In [1]:
!uv add -r requirements.txt

Resolved 284 packages in 3.91s                                                   
⠸ Preparing packages... (0/2)                                                   
⠸ Preparing packages... (0/2)-------------------     0 B/10.84 MiB           
⠼ Preparing packages... (0/2)------------------- 16.00 KiB/10.84 MiB         
⠼ Preparing packages... (0/2)------------------- 16.00 KiB/10.84 MiB         
⠼ Preparing packages... (0/2)------------------- 32.00 KiB/10.84 MiB         
⠼ Preparing packages... (0/2)------------------- 40.34 KiB/10.84 MiB         
⠼ Preparing packages... (0/2)------------------- 40.34 KiB/10.84 MiB         
psycopg2-binary      ------------------------------     0 B/3.12 MiB
⠼ Preparing packages... (0/2)------------------- 40.34 KiB/10.84 MiB         
psycopg2-binary      ------------------------------     0 B/3.12 MiB
⠼ Preparing packages... (0/2)------------------- 42.73 KiB/10.84 MiB         
psycopg2-binary      ------------------------------ 5.36 KiB/3.12 MiB
⠼ Pre

### LLM Evaluation

Read the ingested companies from Postgres (`ventures_db.yc_oss`, loaded by the `yc_oss_to_ventures_db` Kestra flow)
> Get 1/10th of all records for generating ground truths

In [ ]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5440,
    dbname="ventures_db",
    user="postgres",
    password="postgres",
)

df = pd.read_sql("""
    SELECT 
        id,
        name, 
        locations,
        long_description,
        one_liner,
        array_to_string(tags, ', ') AS tags_txt,
        array_to_string(industries, ', ') AS industries_txt,
        subindustry,
        is_hiring::varchar,
        top_company::varchar,
        non_profit::varchar,
        team_size,
        array_to_string(regions, ', ') AS regions_txt,
        stage,
        batch,
        status
    FROM yc_oss
    where random() < 0.1
    """
, conn)

conn.close()

print(f"Loaded {len(df)} companies")
df.head()

Loaded 527 companies


/var/folders/m4/y4ly49q978l8dsj7_7n8ts_40000gp/T/ipykernel_85135/3088500310.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


,id,name,locations,long_description,one_liner,tags_txt,industries_txt,subindustry,is_hiring,top_company,non_profit,team_size,regions_txt,stage,batch,status
0,29152,Cembla,"San Francisco, CA, USA",,AI for legal & government,"SaaS, GovTech, B2B, Legal","B2B, Legal",B2B -> Legal,false,false,false,2.0,"United States of America, America / Canada",Early,Summer 2023,Active
1,12728,MyPetrolPump,"Bengaluru, KA, India",Why did we pick this idea to work on? \r\nIndi...,On demand fuel delivery service in India,"Logistics, India","B2B, Supply Chain and Logistics",B2B -> Supply Chain and Logistics,false,false,false,42.0,"India, South Asia",Early,Summer 2019,Acquired
2,30591,Autonomous Technologies Group,"New York City, NY, USA",Autonomous is a superintelligent financial adv...,Superintelligent financial advisor,"Artificial Intelligence, Consumer Finance",Fintech,Fintech,true,false,false,8.0,"United States of America, America / Canada, Re...",Early,Fall 2025,Active
3,30940,Everest,"San Francisco, CA, USA",,,B2B,Consumer,Consumer,false,false,false,NaN,"United States of America, America / Canada",Early,Fall 2025,Active
4,32230,Codag,"San Francisco, CA, USA",,Log compression for agents.,Developer Tools,"B2B, Infrastructure",B2B -> Infrastructure,false,false,false,1.0,"United States of America, America / Canada",Early,Summer 2026,Active


#### Generating Ground Truth

In [ ]:
# CHooi